# Fairness & Bias Audit — Employee Turnover Model (v1 vs v2)

Notebook ini mengaudit **keadilan (fairness)** model prediksi turnover sebelum
dipakai untuk keputusan yang berdampak pada karyawan.

## Kenapa ini wajib

Alat yang membantu keputusan ketenagakerjaan termasuk **high-risk AI**:

- **EU AI Act** (Regulation 2024/1689, Annex III) — rekrutmen & manajemen pekerja = high-risk.
- **NYC Local Law 144** — wajib audit bias tahunan untuk automated employment decision tools.
- **UU PDP No. 27/2022** (Indonesia) — perlindungan data pribadi karyawan.

## Yang diukur

| Kategori | Metrik |
|---|---|
| Distribusi keputusan | selection rate per kelompok |
| Kesalahan | TPR (recall), FPR, precision per kelompok |
| Kesetaraan | demographic parity difference, equal opportunity difference, equalized odds difference |
| Dampak | impact ratio (aturan **four-fifths** / 80%) |

## Kelompok yang diuji

`usia` (binned), `departemen`, `status_kerja`, `pendidikan`.

> **Catatan penting:** dataset ini **tidak punya atribut gender/etnis**, jadi audit
> ini terbatas pada atribut yang tersedia. Model v1 & v2 juga dilatih ulang pada
> seluruh data, sehingga evaluasi ini bersifat *in-sample* — ini keterbatasan yang
> harus dicatat, bukan diabaikan.

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

CANDIDATE_PATHS = ["hr_turnover.csv", "../data/hr_turnover.csv", "data/hr_turnover.csv"]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("hr_turnover.csv tidak ditemukan.")

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
MODELS_DIR = PROJECT_ROOT / "models"

REF_DATE = pd.Timestamp("2025-01-01")
TARGET = "target_turnover"

NUM_COLS = ["usia", "masa_kerja_bulan", "jam_lembur_bulanan", "log_gaji",
            "usia_saat_masuk", "ada_promosi", "bulan_sejak_promosi", "bulan_hingga_promosi"]
CAT_COLS = ["departemen", "status_kerja", "pendidikan"]
FEATURES = NUM_COLS + CAT_COLS

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)
print("Data :", DATA_PATH)
print("Model:", MODELS_DIR)

Data : data/hr_turnover.csv
Model: /workspace/project/hr-intelligence-system/models


## 1. Feature Engineering (kontrak sama dengan training & API)

`buat_fitur()` **identik** dengan notebook training dan `hr_intelligence_api`.

In [2]:
def buat_fitur(d, ref_date=REF_DATE):
    d = d.copy()
    masuk = pd.to_datetime(d["tanggal_masuk"])
    promosi = pd.to_datetime(d["tanggal_promosi"])
    d["ada_promosi"] = promosi.notna().astype(float)
    d["bulan_sejak_promosi"] = (ref_date - promosi).dt.days / 30.44
    d["bulan_hingga_promosi"] = (promosi - masuk).dt.days / 30.44
    d["usia_saat_masuk"] = d["usia"] - d["masa_kerja_bulan"] / 12
    d["log_gaji"] = np.log1p(d["gaji_bulanan"])
    f = d[FEATURES].copy()
    f[NUM_COLS] = f[NUM_COLS].astype(float)
    return f


df = pd.read_csv(DATA_PATH, parse_dates=["tanggal_masuk", "tanggal_promosi"])
X = buat_fitur(df)
y = df[TARGET].astype(int)
print("Baris:", len(df), "| rasio turnover:", round(float(y.mean()), 4))

Baris: 1000 | rasio turnover: 0.356


## 2. Muat Model v1 & v2

Kedua artifact dimuat dan menghasilkan probabilitas pada data yang sama,
sehingga perbandingan fairness-nya apple-to-apple.

In [3]:
def load_artifact(version):
    path = MODELS_DIR / version / "turnover_model.joblib"
    meta_path = MODELS_DIR / version / "model_metadata.json"
    model = joblib.load(path)
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    return model, meta


models = {}
for version in ["v1", "v2"]:
    m, meta = load_artifact(version)
    thr = float(meta.get("threshold", 0.5))
    proba = m.predict_proba(X)[:, 1]
    models[version] = {
        "model": m,
        "meta": meta,
        "name": meta.get("model", "unknown"),
        "threshold": thr,
        "proba": proba,
        "pred": (proba >= thr).astype(int),
    }
    print(f"{version}: {models[version]['name']:<38} threshold={thr:.2f}")

v1: Logistic Regression                    threshold=0.33


v2: Soft Voting (ensemble)                 threshold=0.33


## 3. Definisi Kelompok (Protected Groups)

Kelompok dibentuk dari atribut yang tersedia. `usia` dipecah menjadi beberapa
bucket agar setiap kelompok punya ukuran sampel memadai.

In [4]:
df_eval = df.copy()
df_eval["kelompok_usia"] = pd.cut(
    df_eval["usia"], bins=[0, 25, 30, 35, 40, 100],
    labels=["<=25", "26-30", "31-35", "36-40", ">40"])

GROUP_COLS = ["kelompok_usia", "departemen", "status_kerja", "pendidikan"]

for col in GROUP_COLS:
    vc = df_eval[col].astype(str).value_counts()
    print(f"{col}: {dict(vc)}")

kelompok_usia: {'31-35': np.int64(252), '36-40': np.int64(228), '>40': np.int64(219), '26-30': np.int64(182), '<=25': np.int64(119)}
departemen: {'Sales': np.int64(148), 'Marketing': np.int64(146), 'HR': np.int64(145), 'Operations': np.int64(145), 'IT': np.int64(141), 'Finance': np.int64(140), 'Legal': np.int64(135)}
status_kerja: {'Tetap': np.int64(571), 'Kontrak': np.int64(302), 'Freelance': np.int64(127)}
pendidikan: {'S1': np.int64(347), 'SMA': np.int64(340), 'Diploma': np.int64(233), 'S2': np.int64(80)}


## 4. Metrik per Kelompok

Untuk setiap kelompok dihitung:

- **n** — jumlah anggota
- **selection rate** — proporsi yang diprediksi keluar (`pred = 1`)
- **TPR / recall** — dari yang benar-benar keluar, berapa yang terdeteksi
- **FPR** — dari yang benar-benar bertahan, berapa yang salah ditandai
- **precision**
- **ROC-AUC**

In [5]:
from sklearn.metrics import roc_auc_score, confusion_matrix


def metrics_per_group(pred, proba, y_true, groups):
    rows = []
    for g in sorted(pd.Series(groups).astype(str).unique()):
        mask = pd.Series(groups).astype(str).values == g
        yt, pr, pb = y_true[mask], pred[mask], proba[mask]
        tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) else np.nan
        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        prec = tp / (tp + fp) if (tp + fp) else np.nan
        auc = roc_auc_score(yt, pb) if len(np.unique(yt)) > 1 else np.nan
        rows.append({
            "kelompok": g, "n": int(mask.sum()),
            "selection_rate": float(pr.mean()),
            "TPR": tpr, "FPR": fpr, "precision": prec, "roc_auc": auc,
        })
    return pd.DataFrame(rows).set_index("kelompok").round(4)


y_np = y.to_numpy()
tabel = {}
for version in ["v1", "v2"]:
    tabel[version] = {
        col: metrics_per_group(models[version]["pred"], models[version]["proba"], y_np, df_eval[col])
        for col in GROUP_COLS
    }

print("=== v2 — metrik per kelompok_usia ===")
print(tabel["v2"]["kelompok_usia"].to_string())
print("\n=== v2 — metrik per departemen ===")
print(tabel["v2"]["departemen"].to_string())

=== v2 — metrik per kelompok_usia ===
            n  selection_rate     TPR     FPR  precision  roc_auc
kelompok                                                         
26-30     182          0.4341  0.6613  0.3167     0.5190   0.7593
31-35     252          0.4603  0.7326  0.3193     0.5431   0.7684
36-40     228          0.4737  0.6842  0.3684     0.4815   0.7385
<=25      119          0.4790  0.8182  0.2800     0.6316   0.8409
>40       219          0.5799  0.7386  0.4733     0.5118   0.7340

=== v2 — metrik per departemen ===
              n  selection_rate     TPR     FPR  precision  roc_auc
kelompok                                                           
Finance     140          0.4000  0.5682  0.3229     0.4464   0.6948
HR          145          0.4483  0.6863  0.3191     0.5385   0.7603
IT          141          0.5248  0.8235  0.3556     0.5676   0.8292
Legal       135          0.5037  0.7209  0.4022     0.4559   0.7503
Marketing   146          0.4521  0.7451  0.2947     0.57

## 5. Metrik Fairness

| Metrik | Arti | Ideal |
|---|---|---|
| Demographic parity difference | selisih selection rate antar kelompok | mendekati 0 |
| Impact ratio (four-fifths) | min/max selection rate | >= 0,80 |
| Equal opportunity difference | selisih TPR antar kelompok | mendekati 0 |
| Equalized odds difference | maks(selisih TPR, selisih FPR) | mendekati 0 |

In [6]:
def fairness_metrics(dfg):
    sr = dfg["selection_rate"]
    tpr = dfg["TPR"].dropna()
    fpr = dfg["FPR"].dropna()
    dp_diff = float(sr.max() - sr.min())
    impact_ratio = float(sr.min() / sr.max()) if sr.max() > 0 else np.nan
    eo_diff = float(tpr.max() - tpr.min()) if len(tpr) else np.nan
    eodds_diff = max(
        float(tpr.max() - tpr.min()) if len(tpr) else 0.0,
        float(fpr.max() - fpr.min()) if len(fpr) else 0.0,
    )
    return {
        "demographic_parity_diff": round(dp_diff, 4),
        "impact_ratio": round(impact_ratio, 4),
        "equal_opportunity_diff": round(eo_diff, 4),
        "equalized_odds_diff": round(eodds_diff, 4),
    }


fair = {}
for version in ["v1", "v2"]:
    fair[version] = pd.DataFrame({
        col: fairness_metrics(tabel[version][col]) for col in GROUP_COLS
    }).T
    fair[version]["lolos_four_fifths"] = fair[version]["impact_ratio"] >= 0.80

for version in ["v1", "v2"]:
    print(f"=== Fairness — {version} ({models[version]['name']}) ===")
    print(fair[version].to_string())
    print()

=== Fairness — v1 (Logistic Regression) ===
               demographic_parity_diff  impact_ratio  equal_opportunity_diff  equalized_odds_diff  lolos_four_fifths
kelompok_usia                   0.1376        0.7570                  0.0815               0.2142              False
departemen                      0.0896        0.8376                  0.2388               0.2388               True
status_kerja                    0.2604        0.6155                  0.1853               0.2411              False
pendidikan                      0.2731        0.5786                  0.2622               0.2622              False

=== Fairness — v2 (Soft Voting (ensemble)) ===
               demographic_parity_diff  impact_ratio  equal_opportunity_diff  equalized_odds_diff  lolos_four_fifths
kelompok_usia                   0.1458        0.7486                  0.1569               0.1933              False
departemen                      0.1473        0.7309                  0.2553             

## 6. Visualisasi

### 6.1 Selection rate per kelompok

Garis putus = aturan four-fifths (80% dari kelompok dengan selection rate tertinggi).

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), GROUP_COLS):
    d = tabel["v2"][col]["selection_rate"].sort_values(ascending=False)
    bars = ax.bar(d.index.astype(str), d.values, color="#4C78A8")
    ax.axhline(d.max() * 0.8, ls="--", color="crimson", lw=1,
               label="ambang four-fifths (80%)")
    for b, v in zip(bars, d.values):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.0%}",
                ha="center", va="bottom", fontsize=9)
    ax.set_title(f"Selection rate per {col} (v2)")
    ax.set_ylabel("proporsi diprediksi keluar")
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

### 6.2 TPR & FPR per kelompok (v2)

Kesenjangan TPR/FPR antar kelompok adalah inti *equalized odds*.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, metric in zip(axes, ["TPR", "FPR"]):
    d = tabel["v2"]["kelompok_usia"][metric]
    ax.bar(d.index.astype(str), d.values, color="#F58518" if metric == "TPR" else "#E45756")
    for i, v in enumerate(d.values):
        ax.text(i, v + 0.01, f"{v:.0%}", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{metric} per kelompok usia (v2)")
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 7. Kesimpulan & Rekomendasi

Output di bawah merangkum temuan utama dan tindakan yang disarankan.

In [9]:
print("=" * 68)
print("RINGKASAN AUDIT FAIRNESS — TURNOVER MODEL")
print("=" * 68)
for version in ["v1", "v2"]:
    f = fair[version]
    gagal = f[~f["lolos_four_fifths"]].index.tolist()
    print(f"\n{version} ({models[version]['name']})")
    print(f"  impact_ratio terendah      : {f['impact_ratio'].min():.3f}")
    print(f"  demographic parity diff max: {f['demographic_parity_diff'].max():.3f}")
    print(f"  equal opportunity diff max : {f['equal_opportunity_diff'].max():.3f}")
    print(f"  atribut gagal four-fifths  : {gagal if gagal else 'tidak ada'}")

print("\nCatatan: audit ini in-sample dan tanpa atribut gender/etnis.")
print("Sebelum produksi: ulangi dengan data held-out + atribut sensitif lengkap,")
print("dan libatkan human review (lihat USE_CASES.md #16).")

RINGKASAN AUDIT FAIRNESS — TURNOVER MODEL

v1 (Logistic Regression)
  impact_ratio terendah      : 0.579
  demographic parity diff max: 0.273
  equal opportunity diff max : 0.262
  atribut gagal four-fifths  : ['kelompok_usia', 'status_kerja', 'pendidikan']

v2 (Soft Voting (ensemble))
  impact_ratio terendah      : 0.452
  demographic parity diff max: 0.410
  equal opportunity diff max : 0.306
  atribut gagal four-fifths  : ['kelompok_usia', 'departemen', 'status_kerja', 'pendidikan']

Catatan: audit ini in-sample dan tanpa atribut gender/etnis.
Sebelum produksi: ulangi dengan data held-out + atribut sensitif lengkap,
dan libatkan human review (lihat USE_CASES.md #16).


In [10]:
report = {
    "in_sample": True,
    "atribut_diuji": GROUP_COLS,
    "catatan": "Dataset tanpa gender/etnis; model refit pada seluruh data.",
    "hasil": {
        version: {
            "model": models[version]["name"],
            "threshold": models[version]["threshold"],
            "fairness": fair[version].drop(columns=["lolos_four_fifths"]).to_dict(orient="index"),
        }
        for version in ["v1", "v2"]
    },
}
out_path = MODELS_DIR / "fairness_report.json"
out_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Laporan disimpan:", out_path)

Laporan disimpan: /workspace/project/hr-intelligence-system/models/fairness_report.json
